<a href="https://colab.research.google.com/github/lucasdieptin/ggdrive/blob/main/Copy_Folder_Google_Drive_to_Google_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder Google Drive to Google Drive - 1TouchPro

In [1]:
#@title Input
from ipywidgets import widgets

dest_text = widgets.Text(description="Your drive", placeholder='Nhập đường link folder Google Drive của bạn')
source_text = widgets.Text(description="Shared drive", placeholder='Nhập đường link folder Google Drive shared')
from_page_text = widgets.Text(description="Từ trang", value="0")
to_page_text = widgets.Text(description="Đến trang", value="0")
max_download_size_text = widgets.Text(description="Tổng dung lượng tối đa(GB)", value="700")
exclude_str_text = widgets.Text(description="Bỏ file, folder có chứa nội dung", value="")

display(dest_text)
display(source_text)
display(from_page_text)
display(to_page_text)
display(max_download_size_text)
display(exclude_str_text)

Text(value='', description='Your drive', placeholder='Nhập đường link folder Google Drive của bạn')

Text(value='', description='Shared drive', placeholder='Nhập đường link folder Google Drive shared')

Text(value='0', description='Từ trang')

Text(value='0', description='Đến trang')

Text(value='700', description='Tổng dung lượng tối đa(GB)')

Text(value='', description='Bỏ file, folder có chứa nội dung')

In [10]:
#@title 🚀 Google Drive Copy Tool (Full Version)
import os
import time
import re
import sys
from googleapiclient.discovery import build
from google.colab import auth
from ipywidgets import widgets

# --- GIAO DIỆN NHẬP LIỆU ---
dest_text = widgets.Text(description="Drive đích", placeholder='Link folder Drive của bạn')
source_text = widgets.Text(description="Drive nguồn", placeholder='Link folder Drive shared')
from_page_text = widgets.Text(description="Từ trang", value="0")
to_page_text = widgets.Text(description="Đến trang", value="0")
max_download_size_text = widgets.Text(description="Giới hạn (GB)", value="700")
exclude_str_text = widgets.Text(description="Bỏ qua (vd: .txt)", value="")

display(dest_text, source_text, from_page_text, to_page_text, max_download_size_text, exclude_str_text)

# --- CLASS XỬ LÝ CHÍNH ---
class DownloadFromDrive:
    def __init__(self):
        self._total_size = 0
        self._limit_size = 0
        self.excluded_strings = []

    def get_user_credential(self):
        auth.authenticate_user()
        return build('drive', 'v3')

    def check_file_info(self, drive_service, dest_folder_id, name):
        """Kiểm tra chi tiết file: trả về info nếu tồn tại chính xác tên"""
        try:
            processed_name = name.replace("'", "\\'")
            query = f"'{dest_folder_id}' in parents and name = '{processed_name}' and trashed = false"
            results = drive_service.files().list(
                q=query,
                fields='files(id, name, size, modifiedTime)',
                supportsAllDrives=True,
                includeItemsFromAllDrives=True).execute()

            files = results.get('files', [])
            return files[0] if files else None
        except:
            return None

    def get_unused_name(self, drive_service, dest_folder_id, original_name):
        """Nếu file khác nội dung, tìm tên mới dạng File_1.ext, File_2.ext..."""
        name_only, ext = os.path.splitext(original_name)
        counter = 1
        new_name = original_name
        while self.check_file_info(drive_service, dest_folder_id, new_name):
            new_name = f"{name_only}_{counter}{ext}"
            counter += 1
        return new_name

    def get_childs_from_folder(self, drive_service, folder_id, from_page, to_page):
        files = []
        page_token = None
        query = f"'{folder_id}' in parents and trashed = false"
        if self.excluded_strings:
            not_contains = " and ".join([f"not name contains '{ext}'" for ext in self.excluded_strings])
            query += f" and ({not_contains})"

        pages = 0
        while True:
            pages += 1
            response = drive_service.files().list(
                q=query,
                orderBy='name, createdTime',
                fields='files(id, name, mimeType, size, modifiedTime), nextPageToken',
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True).execute()

            if (from_page < pages <= to_page) or to_page == 0:
                files.extend(response.get('files', []))

            page_token = response.get('nextPageToken')
            if not page_token or (pages >= to_page > 0): break
        return files

    def copy_file(self, drive_service, dest_folder_id, source_file):
        # Nếu là Folder
        if source_file['mimeType'] == 'application/vnd.google-apps.folder':
            print(f"\n📂 Đang xử lý thư mục: {source_file['name']}")
            sub_folder_id = self.create_folder(drive_service, dest_folder_id, source_file['name'])
            child_files = self.get_childs_from_folder(drive_service, source_file['id'], 0, 0)
            self.copy_multiple_files(drive_service, sub_folder_id, child_files)
            return

        # Nếu là File: Kiểm tra Verify
        existing_file = self.check_file_info(drive_service, dest_folder_id, source_file['name'])
        target_name = source_file['name']
        should_copy = True

        if existing_file:
            src_size = source_file.get('size')
            dst_size = existing_file.get('size')
            src_time = source_file.get('modifiedTime')
            dst_time = existing_file.get('modifiedTime')

            # So sánh chính xác dung lượng và ngày cập nhật
            if src_size == dst_size and src_time == dst_time:
                print(f"[-] {source_file['name']} đã tồn tại & khớp dữ liệu. Bỏ qua.")
                should_copy = False
            else:
                target_name = self.get_unused_name(drive_service, dest_folder_id, source_file['name'])
                print(f"[!] {source_file['name']} có sự khác biệt. Copy bản mới: {target_name}")

        if should_copy:
            try:
                start_time = time.time()
                body = {'parents': [dest_folder_id], 'name': target_name}
                drive_service.files().copy(fileId=source_file['id'], body=body, supportsAllDrives=True).execute()

                size_mb = int(source_file.get('size', 0)) / (1024 * 1024)
                self._total_size += size_mb
                print(f"[+] Thành công: {target_name} ({size_mb:0.2f} MB)")

                if self._total_size >= (self._limit_size * 1024):
                    print(f"❌ Đã đạt giới hạn {self._limit_size} GB. Dừng chương trình.")
                    sys.exit()
            except Exception as e:
                print(f"❌ Lỗi copy {target_name}: {e}")

    def create_folder(self, drive_service, dest_folder_id, name):
        exist = self.check_file_info(drive_service, dest_folder_id, name)
        if exist: return exist['id']
        body = {'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [dest_folder_id]}
        return drive_service.files().create(body=body, fields='id', supportsAllDrives=True).execute()['id']

    def copy_multiple_files(self, drive_service, dest_folder_id, source_files):
        for f in source_files:
            self.copy_file(drive_service, dest_folder_id, f)

    def extract_id(self, url):
        match = re.search(r'[-\w]{25,}', url)
        return match.group(0) if match else None

    def run(self, dest_l, src_l, f_pg, t_pg):
        service = self.get_user_credential()
        d_id = self.extract_id(dest_l)
        s_id = self.extract_id(src_l)

        if not d_id or not s_id:
            print("❌ Không nhận diện được Link Drive!")
            return

        src_info = service.files().get(fileId=s_id, supportsAllDrives=True).execute()
        print(f"🏁 Bắt đầu copy từ: {src_info['name']}")

        new_dest_id = self.create_folder(service, d_id, src_info['name'])
        all_files = self.get_childs_from_folder(service, s_id, f_pg, t_pg)

        start_t = time.time()
        self.copy_multiple_files(service, new_dest_id, all_files)
        end_t = time.time()

        print(f"\n✅ HOÀN THÀNH!")
        print(f"📊 Tổng dung lượng: {self._total_size/1024:0.2f} GB")
        print(f"⏱️ Thời gian: {int(end_t - start_t)} giây")

# --- KÍCH HOẠT ---
def start_process():
    downloader = DownloadFromDrive()
    downloader._limit_size = float(max_download_size_text.value)
    downloader.excluded_strings = [ext.strip() for ext in exclude_str_text.value.split(",") if ext.strip()]
    downloader.run(dest_text.value, source_text.value, int(from_page_text.value), int(to_page_text.value))

# Nút bấm để chạy
button = widgets.Button(description="BẮT ĐẦU COPY", button_style='success')
button.on_click(lambda x: start_process())
display(button)

Text(value='', description='Drive đích', placeholder='Link folder Drive của bạn')

Text(value='', description='Drive nguồn', placeholder='Link folder Drive shared')

Text(value='0', description='Từ trang')

Text(value='0', description='Đến trang')

Text(value='700', description='Giới hạn (GB)')

Text(value='', description='Bỏ qua (vd: .txt)')

Button(button_style='success', description='BẮT ĐẦU COPY', style=ButtonStyle())

🏁 Bắt đầu copy từ: XU LY BAO BI
[+] Thành công: baitang 15-10-25.ai (721.47 MB)
[+] Thành công: baitang_16-16-8_50kg-2.ai (4.64 MB)

📂 Đang xử lý thư mục: EXPORT
[+] Thành công: baitang 15-10-25.png (17.84 MB)
[+] Thành công: baitang_16-16-8_50kg-2.png (4.76 MB)
[+] Thành công: KALI MIENG DO_CPC_50KG_BAITANG_20260209.png (3.29 MB)
[+] Thành công: LA554296 - C22-4-22.png (17.50 MB)
[+] Thành công: NPK BAITANG_15-15-15_50kg.png (5.02 MB)
[+] Thành công: NPK BAITANG_20-20-15_50kg.png (4.90 MB)
[+] Thành công: UREA-CAMAU_CPC_50KG_BAITANG.png (2.94 MB)
[+] Thành công: KALI MIENG DO_CPC_50KG_BAITANG_20260209.ai (4.21 MB)
[+] Thành công: LA554296 - C22-4-22.ai (721.48 MB)
[+] Thành công: NPK BAITANG_15-15-15_50kg.pdf (3.90 MB)
[+] Thành công: NPK BAITANG_20-20-15_50kg.pdf (4.02 MB)
[+] Thành công: PHOSAGRO DAP (SSG DAP)_NAUDEN_50KG-v20251231-v2B2 vien ngoai - Copy.ai (3.85 MB)

📂 Đang xử lý thư mục: PNG PDF
[+] Thành công: baitang 15-10-25.pdf (5.85 MB)
[+] Thành công: baitang_16-16-8_50kg-2.